# Numerai Model Training & Submission Pipeline

This notebook is a self-contained environment for training, validating, and submitting models to Numerai. It consolidates all the logic from the `local/` scripts into a singular, top-to-bottom executable pipeline.

### Project Features
- **Data Management**: Automated download of the latest Numerai datasets.
- **Exploration**: Insights into feature distributions and target correlations.
- **Modeling**: Support for LightGBM, Neural Networks, and Transformers.
- **Validation**: Comprehensive performance metrics (CORR, MMC, Sharpe, Drawdown).
- **Submission**: Robust ensembling and pickling for Numerai model uploads.


## 1. Environment Setup
Install required dependencies for the Numerai ecosystem.


In [ ]:
!pip install -q numerapi numerai-tools lightgbm tensorflow cloudpickle pandas matplotlib scikit-learn scipy pyarrow


## 2. Configuration & Initialization
Define the data version and target settings.


In [ ]:
import json
import os
from pathlib import Path
from numerapi import NumerAPI

DATA_VERSION = 'v5.2'
napi = NumerAPI()

# Standard Numerai Target (can be modified to use auxiliary targets like target_ender_20)
TARGETS = ['target']
MAIN_TARGET = TARGETS[0]

print(f'Using Data Version: {DATA_VERSION}')


## 3. Dataset Download
Fetch the training and validation datasets.


In [ ]:
# Download features metadata
napi.download_dataset(f'{DATA_VERSION}/features.json')
with open(f'{DATA_VERSION}/features.json') as f:
    feature_metadata = json.load(f)

feature_cols = feature_metadata['feature_sets']['medium'] # Using medium by default
target_cols = feature_metadata['targets']

# Download training and validation data (this might take a few minutes)
print('Downloading datasets...')
napi.download_dataset(f'{DATA_VERSION}/train.parquet')
napi.download_dataset(f'{DATA_VERSION}/validation.parquet')
print('Download complete.')


## 4. Exploratory Data Analysis (EDA)
Analyze target correlations and data distribution.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load a sample for EDA
train_sample = pd.read_parquet(f'{DATA_VERSION}/train.parquet', columns=['era'] + target_cols).head(100000)

# Visualize correlations
corrs = train_sample[target_cols].corrwith(train_sample[MAIN_TARGET]).sort_values(ascending=False)
corrs.head(20).plot(kind='bar', figsize=(10, 5), title=f'Top Correlations with {MAIN_TARGET}')
plt.show()

# Era distribution
train_sample.groupby('era').size().plot(title='Number of rows per era (Sample)')
plt.show()


## 5. Model Training
We will train three types of models: LightGBM, a Simple Neural Network, and a Transformer-based model.


### 5.1 LightGBM Training


In [ ]:
import lightgbm as lgb
import pickle

LGBM_TARGET = TARGETS[0]

# --- More aggressive memory optimization for training data loading ---
# 1. Load only 'era' column to identify which eras to use for subsetting
era_df = pd.read_parquet(f'{DATA_VERSION}/train.parquet', columns=['era'])
all_eras = sorted(era_df['era'].unique())

# Identify eras for the subset (every 4th era)
subset_eras = all_eras[::4]

# 2. Load only the relevant eras and columns directly
print(f'Loading training data for {len(subset_eras)} eras...')
train = pd.read_parquet(
    f'{DATA_VERSION}/train.parquet',
    columns=['era'] + feature_cols + [MAIN_TARGET],
    filters=[('era', 'in', subset_eras)] # Load only the subsetted eras
)

# Memory optimization: Downcast dtypes after loading the filtered data
train['era'] = train['era'].astype('category')
for col in feature_cols:
    train[col] = train[col].astype('float32')
train[MAIN_TARGET] = train[MAIN_TARGET].astype('float32')

# split eras into train and validation sets (we'll use the validation set for early stopping)
eras = sorted(train["era"].unique())
cut = int(len(eras) * 0.8)

train_eras = eras[:cut]
valid_eras = eras[cut:]

train_mask = train["era"].isin(train_eras)
valid_mask = train["era"].isin(valid_eras)

print(f'Training LightGBM on {len(train)} rows...')
lgbm_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=5,
    num_leaves=2**5-1,
    colsample_bytree=0.1,
    extra_trees = True,
    device_type = "gpu",
    random_state=42
)

lgbm_model.fit(
    train.loc[train_mask, feature_cols],
    train.loc[train_mask,   LGBM_TARGET],
    eval_set=[(
        train.loc[valid_mask, feature_cols],
        train.loc[valid_mask, LGBM_TARGET]
    )],
    callbacks=[lgb.early_stopping(200), lgb.log_evaluation(100)]
)

os.makedirs('models', exist_ok=True)
with open('models/lgbm_models.pkl', 'wb') as f:
    pickle.dump({LGBM_TARGET: lgbm_model}, f)
print('LightGBM model saved.')

### 5.2 Neural Network Training


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# Check for GPU availability
if tf.config.list_physical_devices('GPU'):
    print("GPU is available. Training will run on GPU.")
else:
    print("GPU is not available. Training will run on CPU.")

def create_nn_model(input_shape):
    # Keras models automatically use GPU if available and configured
    model = models.Sequential([
        layers.Input(shape=(input_shape,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

nn_model = create_nn_model(len(feature_cols))
print('Training Neural Network...')

# Prepare data for Neural Network (convert to numpy arrays for TensorFlow)
X_train_nn = train.loc[train_mask, feature_cols].values.astype(np.float32)
y_train_nn = train.loc[train_mask, MAIN_TARGET].values.astype(np.float32)
X_valid_nn = train.loc[valid_mask, feature_cols].values.astype(np.float32)
y_valid_nn = train.loc[valid_mask, MAIN_TARGET].values.astype(np.float32)

nn_model.fit(
    X_train_nn,
    y_train_nn,
    epochs=30,
    batch_size=256,
    verbose=1,
    validation_data=(X_valid_nn, y_valid_nn)
)
nn_model.save('models/nn_model.keras')


### 5.3 Transformer Training


In [ ]:
class FeatureEmbedding(layers.Layer):
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
    def build(self, input_shape):
        self.projection = layers.Dense(self.d_model)
    def call(self, x):
        x = tf.expand_dims(x, axis=-1)
        return self.projection(x)
    def get_config(self):
        config = super().get_config()
        config.update({'d_model': self.d_model})
        return config

class LinformerAttention(layers.Layer):
    def __init__(self, d_model, num_heads, k, **kwargs):
        super().__init__(**kwargs)
        self.d_model, self.num_heads, self.k = d_model, num_heads, k
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
    
    def build(self, input_shape):
        seq_len = input_shape[1]
        self.E = self.add_weight(shape=(seq_len, self.k), initializer='glorot_uniform', trainable=True, name='E')
        self.F = self.add_weight(shape=(seq_len, self.k), initializer='glorot_uniform', trainable=True, name='F')

    def call(self, x, training=False):
        k_proj = tf.einsum('bsd,sk->bkd', x, self.E)
        v_proj = tf.einsum('bsd,sk->bkd', x, self.F)
        return self.mha(x, v_proj, k_proj, training=training)

    def get_config(self):
        config = super().get_config()
        config.update({'d_model': self.d_model, 'num_heads': self.num_heads, 'k': self.k})
        return config

class TransformerEncoderBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ffn_dim, dropout_rate, k=None, **kwargs):
        super().__init__(**kwargs)
        self.d_model, self.num_heads, self.ffn_dim, self.dropout_rate, self.k = d_model, num_heads, ffn_dim, dropout_rate, k
        if k:
            self.attn = LinformerAttention(d_model, num_heads, k)
        else:
            self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = models.Sequential([layers.Dense(ffn_dim, activation='relu'), layers.Dense(d_model)])
        self.layernorm1, self.layernorm2 = layers.LayerNormalization(), layers.LayerNormalization()
        self.dropout1, self.dropout2 = layers.Dropout(dropout_rate), layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        if self.k:
            attn_output = self.attn(x, training=training)
        else:
            attn_output = self.attn(x, x, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        x = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(x, training=training)
        ffn_output = self.dropout2(ffn_output, training=training)
        x = self.layernorm2(x + ffn_output)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({'d_model': self.d_model, 'num_heads': self.num_heads, 'ffn_dim': self.ffn_dim, 'dropout_rate': self.dropout_rate, 'k': self.k})
        return config

def create_transformer_model(num_features):
    inputs = layers.Input(shape=(num_features,))
    x = FeatureEmbedding(32)(inputs)
    pos_encoding = tf.Variable(tf.random.normal([1, num_features, 32], stddev=0.02), trainable=True)
    x = x + pos_encoding
    
    # ALBERT-style: Shared parameters across layers
    # Linformer-style: k=32 bottleneck for attention
    shared_encoder = TransformerEncoderBlock(32, 2, 64, 0.1, k=32)
    for _ in range(4): # Increased depth but shared parameters
        x = shared_encoder(x)
        
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(1)(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

transformer_model = create_transformer_model(len(feature_cols))
print('Training Transformer...')

x_train = train.loc[train_mask, feature_cols].values.astype(np.float32)
y_train = train.loc[train_mask, MAIN_TARGET].values.astype(np.float32)
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(5000, reshuffle_each_iteration=True)
train_ds = train_ds.batch(64)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

transformer_model.fit(train_ds, epochs=10, verbose=1)
transformer_model.save('models/transformer_model.keras')


## 6. Validation & Reporting
Evaluate the performance of our models on the validation set.


In [ ]:
from numerai_tools.scoring import numerai_corr, correlation_contribution, neutralize

def compute_perf_metrics(series: pd.Series) -> dict:
    series = pd.Series(np.asarray(series).reshape(-1)).dropna()
    mean = series.mean()
    std = series.std(ddof=0)
    sharpe = mean / std if std != 0 else np.nan
    return {'mean': mean, 'std': std, 'sharpe': sharpe}

# Load validation data
val = pd.read_parquet(f'{DATA_VERSION}/validation.parquet', columns=['era', 'data_type', 'target'] + feature_cols)
val = val[val['data_type'] == 'validation']

# Downsample to eras to reduce memory usage and speedup evaluation
# Default is slower and higher memory usage, but more accurate evaluation.
val = val[val["era"].isin(val["era"].unique()[::4])]

# Eras are 1 week apart, but targets look 20 days (o 4 weeks/eras) into the future,
# so we need to "embargo" the first 4 eras following our last train era to avoid "data leakage"
train = pd.read_parquet(f"{DATA_VERSION}/train.parquet",columns=["era"]) # Load training eras for embargo calculation
last_train_era = int(train["era"].unique()[-1])
eras_to_embargo = [str(era).zfill(4) for era in [last_train_era + i for i in range(4)]]
val = val[~val["era"].isin(eras_to_embargo)]

# Generate model predictions
val['pred_lgbm'] = lgbm_model.predict(val[feature_cols])
val['pred_nn'] = nn_model.predict(val[feature_cols].values.astype(np.float32), verbose=0).reshape(-1)
#val['pred_transformer'] = transformer_model.predict(val[feature_cols].values.astype(np.float32), verbose=0).reshape(-1)

# Rank LGBM predictions per era
val["pred_lgbm_ranked_per_era"] = val.groupby("era")["pred_lgbm"].rank(pct=True)
# Rank NN predictions per era
val["pred_nn_ranked_per_era"] = val.groupby("era")["pred_nn"].rank(pct=True)
# Rank Transformer predictions per era
#val["pred_transformer_ranked_per_era"] = val.groupby("era")["pred_transformer"].rank(pct=True)

# Ensemble: Adjusted weights to include Transformer
val['pred_ensemble'] = (0.85 * val["pred_lgbm_ranked_per_era"]) + \
                       (0.15 * val["pred_nn_ranked_per_era"])
                        #+ \ (0.15 * val["pred_transformer_ranked_per_era"])
val['prediction'] = val.groupby('era')['pred_ensemble'].rank(pct=True)

# Feature neutralize
val["prediction"] = neutralize(
    val[["prediction"]], val[feature_cols], proportion=0.01)["prediction"]

# Final Rerank
val["prediction"] = val.groupby("era")["prediction"].rank(pct=True)

# Compute Metrics
per_era_corr = val.groupby('era').apply(lambda x: numerai_corr(x[['prediction']], x['target']).iloc[0])
print('Validation Metrics (CORR):')
print(pd.DataFrame({'CORR': compute_perf_metrics(per_era_corr)}))

per_era_corr.plot(kind='bar', title='Per-era CORR', figsize=(12, 5))
plt.show()

## 7. Submission Preparation
Finalize the `predict` function and save it as a pickle for Numerai upload.


In [ ]:
import cloudpickle

# Standard Numerai Submission Format
def predict(live_features: pd.DataFrame, live_benchmark_models: pd.DataFrame) -> pd.DataFrame:
    # 1. Feature Selection
    X = live_features[feature_cols]

    # 2. LGBM Prediction
    lgbm_pred = lgbm_model.predict(X)
    lgbm_rank = pd.Series(lgbm_pred, index=live_features.index).rank(pct=True)

    # 3. NN Prediction
    nn_pred = nn_model.predict(X.values.astype(np.float32), verbose=0).reshape(-1)
    nn_rank = pd.Series(nn_pred, index=live_features.index).rank(pct=True)

    # 4. Transformer Prediction
    tran_pred = transformer_model.predict(X.values.astype(np.float32), verbose=0).reshape(-1)
    tran_rank = pd.Series(tran_pred, index=live_features.index).rank(pct=True)

    # 5. Weighted Blend (aligned with validation)
    ensemble = (0.7 * lgbm_rank) + (0.15 * nn_rank) + (0.15 * tran_rank)

    return ensemble.rank(pct=True, method='first').to_frame('prediction')

# Smoke Test
sample_live = val.head(100)
print('Smoke test output:')
print(predict(sample_live, None).head())

# Save for upload
with open('numerai_upload(1).pkl', 'wb') as f:
    f.write(cloudpickle.dumps(predict))
# Download file if running in Google Colab
try:
    from google.colab import files
    files.download('numerai_upload(1).pkl')
except:
    pass
print('Run next cell to save .pkl')

In [ ]:
from google.colab import files
files.download('numerai_upload2.pkl')
print('Successfully saved numerai_upload.pkl')
